# Tutorial 13 — Speech, audio representations, and multilingual ASR

[Lecture notes](../lectures/13_speech.md) · [Exercise sheet](../exercises/13_speech.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Compute an STFT on a synthetic waveform, collapse CTC paths, and implement WER through dynamic programming.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


![Illustration of a local window over a waveform.](../data/audio-window.svg)

Illustration of the default task. If you change parameters, interpret the newly computed outputs rather than this static illustration.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. A synthetic waveform and STFT

The signal is a tone with noise, not speech. An STFT is not yet a mel spectrogram. We preserve the sample rate explicitly.

Before running: predict the output or the invariant being checked.


In [2]:
sr=16000; t=np.arange(sr)/sr
wave=.5*np.sin(2*np.pi*440*t)+.01*rng.normal(size=sr)
frame_length=400; hop=160
frames=np.stack([wave[i:i+frame_length] for i in range(0,len(wave)-frame_length+1,hop)])
spectrum=np.abs(np.fft.rfft(frames*np.hanning(frame_length),axis=1))
log_spectrum=np.log(spectrum+1e-8)
freqs=np.fft.rfftfreq(frame_length,1/sr)
peak=freqs[spectrum.mean(axis=0).argmax()]
print("seconds",len(wave)/sr,"STFT shape",spectrum.shape,"dominant Hz",peak)
assert abs(peak-440)<=sr/frame_length


seconds 1.0 STFT shape (98, 201) dominant Hz 440.0


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Collapse CTC paths

Repeated labels merge before blank removal. Compare adjacent repeated labels with repetitions separated by a blank.

Before running: predict the output or the invariant being checked.


In [3]:
def collapse(path,blank="_"):
    merged=[c for i,c in enumerate(path) if i==0 or c!=path[i-1]]
    return "".join(c for c in merged if c!=blank)
print(collapse(["a","a","_","a","b","b","_"]))
assert collapse(["a","a"])=="a"
assert collapse(["a","_","a"])=="aa"


aab


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. WER through dynamic programming

We compute total edit distance. Empty references are explicitly handled; no automatic punctuation normalization is applied.

Before running: predict the output or the invariant being checked.


In [4]:
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1),dtype=int)
    D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1,D[i,j-1]+1,D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return int(D[-1,-1])
def wer(ref,hyp):
    a,b=ref.split(),hyp.split()
    if not a:
        if not b: return 0.
        raise ValueError("WER undefined for empty reference and nonempty hypothesis")
    return edit_distance(a,b)/len(a)
ref="the lab opens today"; hyp="lab opens on monday"
print("WER",wer(ref,hyp))
assert wer(ref,hyp)==.75
assert wer("yes","yes yes yes")==2.
assert edit_distance("kitten","sitting")==3


WER 0.75


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why can WER exceed 100%?
2. What is wrong with treating a 48 kHz recording as 16 kHz without resampling?
3. Why should two clips from one speaker stay in the same split?


## Extension and submission

Run `extensions/hf_audio.py --audio path.wav` on a permitted short clip. It resamples with librosa and runs a pretrained Whisper pipeline; score the output against a human transcript.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
